# Calibrate for Pre-Emphasis

Compares no, fitting and optimisation pre-emphasis for the gradient and offset.

### Import 

In [1]:
from matipo import GLOBALS_DIR
import panel as pn
import numpy as np
import pandas as pd
import copy
import matplotlib.pyplot as plt
plt.style.use('default')
import yaml
from pathlib import Path
import optuna
import shutil
import nest_asyncio
nest_asyncio.apply()

In [2]:
from user_interface.calibrate import Calibrate
scan_type = "calibrate_ec"
cal = Calibrate(scan_type)

### Set Calibration Scan Parameters

In [ ]:
g_max = 0.2
g_min = -0.2
num_datapoints = 11

g_list = np.linspace(g_min, g_max, num_datapoints)
print(g_list)

In [ ]:
flip_angle = 4.5
axis = "Z"

seq_params = {
    "OPT_AXIS": axis,
    "n_scans": 1,
    "a_pulse": comp.get_yaml_info("hardpulse_90_tx0.yaml")["a_90"],
    "t_pulse": (comp.get_yaml_info("hardpulse_90_tx0.yaml")["t_90"])*flip_angle/90,
    "t_dw": 10e-6,
    "n_samples": 100,
    "t_acq_delay": 100e-6,
    "t_grad": 10.0,
    "t_ramp": 1e-3,
    "t_wait": 8e-3,
    "n_pulses": 30,
    "t_end": 10.0,
    "t_next": 5,  # wait between scans
    "g_list": g_list, # convert this into g_read before using
    "tx_chan": 0,
    "rx_chan_list": (3, 0, 1),
    "pre_emphasis": True,
    "b0": True,
    "file_cals": GLOBALS_DIR,
    "d_r": comp.get_yaml_info("probe_distances.yaml")["d_r"],
    "d_c": comp.get_yaml_info("probe_distances.yaml")["d_c"],
    "d_l": comp.get_yaml_info("probe_distances.yaml")["d_l"],

}

print(seq_params)

### Run Calibration
Saves calibration information into a pkl file locally

In [ ]:
dict_, f_list, name = await cal.run_comparison(self, seq_params)

### Analyse and Fit to Calibration 

In [ ]:
g_hzpm_list, mask, g_vals, g_h = cal.analyse_pkl(name)

slope, intercept = cal.fit_ghzpm_slope(name, g_hzpm_list, mask, g_vals, g_h, g_fit_min, g_fit_max)

### Save Slope into Globals

In [ ]:
#  DO THIS